outras infos de sistema


Databricks SQL
https://www.databricks.com/br/product/pricing/databricks-sql

Preços do Azure Databricks
https://azure.microsoft.com/pt-br/pricing/details/databricks/

Pricing calculator
https://www.databricks.com/br/product/pricing/product-pricing/instance-types


Preços de Máquinas Virtuais do Linux
https://azure.microsoft.com/pt-br/pricing/details/virtual-machines/linux/




## Diferença entre SQL Warehouse (Serverless), Cluster Compute e DBU no Databricks

- **SQL Warehouse (Serverless)**: Serviço gerenciado do Databricks para execução de queries SQL, dashboards e BI, sem necessidade de gerenciar clusters. Recursos são alocados automaticamente conforme a demanda. Pagamento por uso, medido em DBUs/hora.

- **Cluster Compute (Databricks)**: Conjunto de máquinas virtuais dedicadas para processamento de dados, notebooks, jobs e workloads Spark. Você gerencia o tamanho, tipo e tempo de vida do cluster. Pagamento por uso, também medido em DBUs/hora.

- **Como são pagos**:
  - **SQL Warehouse (Serverless)**: Pagamento automático por uso (DBUs/hora), sem necessidade de provisionar ou gerenciar infraestrutura.
  - **Cluster Compute**: Pagamento por uso (DBUs/hora), conforme o tipo e tamanho do cluster criado.

- **O que é DBU?**
  - **DBU (Databricks Unit)**: Unidade de cobrança do Databricks baseada no consumo de recursos computacionais (CPU, RAM, tipo de workload) por hora. O valor em $/DBU varia conforme o tipo de cluster ou serviço utilizado.

In [0]:
%sql
SHOW SCHEMAS IN system

In [0]:
%sql
show tables in system.billing

In [0]:
%sql
select * from system.billing.usage limit 10

# 📚 Guia Completo: System Tables do Databricks

O **System Catalog** (`system`) é um catálogo especial do Unity Catalog que fornece visibilidade completa sobre:
- 💰 **Custos e Billing**
- 🔍 **Auditoria e Compliance**
- ⚡ **Performance e Query Optimization**
- 🖥️ **Recursos Computacionais**
- 🤖 **Machine Learning e AI**
- 🔄 **Pipelines e Jobs**
- 📊 **Linhagem de Dados**

Este guia cobre **todos os 11 schemas** disponíveis com exemplos práticos de uso!

---

## 1️⃣ SYSTEM.BILLING - Gestão de Custos 💰

### Tabelas Disponíveis:
* `system.billing.usage` - Consumo detalhado de DBUs
* `system.billing.list_prices` - Preços oficiais por SKU
* `system.billing.account_prices` - Preços específicos da sua conta

### 📋 Significado:
Monitore e otimize custos do Databricks com dados granulares de consumo por workspace, cluster, usuário, SKU e tipo de workload.

### ✅ Casos de Uso:
1. **Análise de custo por departamento/projeto**
2. **Identificar recursos mais caros**
3. **Criar alertas de budget**
4. **Chargeback interno**
5. **Projeção de custos futuros**

In [0]:
%sql
-- Análise de custo dos últimos 30 dias
SELECT 
  u.workspace_id,
  u.usage_metadata.cluster_id,
  u.sku_name,
  ROUND(SUM(u.usage_quantity), 2) as total_dbus
FROM 
  system.billing.usage u
WHERE 
  u.usage_date >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY 1, 2, 3
ORDER BY total_dbus DESC
LIMIT 10;

## 2️⃣ SYSTEM.ACCESS - Auditoria e Governança 🔐

### Tabelas Disponíveis:
* `system.access.audit` - Log completo de todas as ações no workspace
* `system.access.table_lineage` - Linhagem de tabelas (origem → destino)
* `system.access.column_lineage` - Linhagem em nível de coluna
* `system.access.assistant_events` - Interações com AI Assistant
* `system.access.clean_room_events` - Eventos de Clean Rooms
* `system.access.outbound_network` - Conexões externas do workspace

### 📋 Significado:
Rastreie **TUDO** que acontece no workspace: quem acessou, criou, modificou ou deletou dados, tabelas, notebooks, jobs, etc.

### ✅ Casos de Uso:
1. **Compliance e LGPD/GDPR**
2. **Investigar incidentes de segurança**
3. **Auditoria de acessos a dados sensíveis**
4. **Rastreamento de linhagem completa (data lineage)**
5. **Identificar tabelas órfãs ou não utilizadas**

In [0]:
%sql
-- Auditoria de acesso a tabelas sensíveis
SELECT 
  event_date,
  user_identity.email as usuario,
  action_name,
  request_params.full_name_arg as tabela_acessada,
  request_params.operation as tipo_operacao
FROM 
  system.access.audit
WHERE 
  event_date >= CURRENT_DATE() - INTERVAL 7 DAYS
  AND action_name IN ('getTable', 'readTable', 'createTable', 'deleteTable')
  AND request_params.full_name_arg LIKE '%pii%' -- tabelas com dados sensíveis
ORDER BY event_date DESC
LIMIT 100;

In [0]:
%sql
-- Descobrir origem e destino de uma tabela específica
SELECT 
  source_table_full_name as tabela_origem,
  target_table_full_name as tabela_destino,
  source_type,
  target_type,
  event_time
FROM 
  system.access.table_lineage
WHERE 
  target_table_full_name = 'catalog.schema.minha_tabela'
ORDER BY event_time DESC
LIMIT 50;

## 3️⃣ SYSTEM.QUERY - Histórico e Performance 🚀

### Tabelas Disponíveis:
* `system.query.history` - Histórico completo de todas as queries executadas

### 📋 Significado:
Como você já viu neste notebook, esta tabela registra **TODAS** as queries executadas (SQL, notebooks, dashboards, jobs) com métricas detalhadas de performance.

### ✅ Casos de Uso:
1. **Identificar queries lentas (bottlenecks)**
2. **Análise de uso do cache**
3. **Otimização de performance**
4. **Troubleshooting de falhas**
5. **Monitoramento de SLAs**

In [0]:
%sql
-- Top 20 queries mais lentas com detalhes de otimização
SELECT 
  statement_id,
  executed_by,
  SUBSTRING(statement_text, 1, 100) as query_preview,
  ROUND(total_duration_ms / 1000, 2) as tempo_segundos,
  ROUND(read_bytes / 1024 / 1024, 2) as mb_lidos,
  produced_rows,
  from_result_cache,
  execution_status,
  error_message,
  start_time
FROM 
  system.query.history
WHERE 
  start_time >= CURRENT_DATE() - INTERVAL 7 DAYS
  AND execution_status = 'FINISHED'
  AND total_duration_ms > 10000 -- queries > 10 segundos
ORDER BY total_duration_ms DESC
LIMIT 20;

## 4️⃣ SYSTEM.COMPUTE - Recursos Computacionais 🖥️

### Tabelas Disponíveis:
* `system.compute.clusters` - Informações de todos os clusters
* `system.compute.warehouses` - SQL Warehouses configurados
* `system.compute.warehouse_events` - Eventos de warehouse (start, stop, scale)
* `system.compute.node_timeline` - Timeline de utilização de nós
* `system.compute.node_types` - Tipos de instâncias disponíveis

### 📋 Significado:
Monitore clusters, SQL warehouses e recursos computacionais: configuração, estado, eventos de lifecycle e custos associados.

### ✅ Casos de Uso:
1. **Otimizar tamanho de clusters**
2. **Identificar clusters ociosos**
3. **Monitorar autoscaling**
4. **Análise de uptime vs downtime**
5. **Planejamento de capacidade**

In [0]:
%sql
-- Análise de clusters e custos
SELECT 
  cluster_id,
  cluster_name,
  driver_node_type,
  worker_node_type,
  worker_count,
  min_autoscale_workers,
  max_autoscale_workers,
  owned_by,
  create_time,
  DATEDIFF(hour, create_time, CURRENT_TIMESTAMP()) as horas_uptime
FROM 
  system.compute.clusters
WHERE 
  delete_time IS NULL
ORDER BY create_time DESC;

In [0]:
%sql
-- Monitorar starts/stops de SQL Warehouse
SELECT 
  warehouse_id,
  event_type,
  event_time,
  cluster_count
FROM 
  system.compute.warehouse_events
WHERE 
  event_time >= CURRENT_DATE() - INTERVAL 7 DAYS
ORDER BY event_time DESC
LIMIT 100;

## 5️⃣ SYSTEM.LAKEFLOW - Jobs e Pipelines 🔄

### Tabelas Disponíveis:
* `system.lakeflow.jobs` - Configuração de todos os jobs
* `system.lakeflow.job_run_timeline` - Histórico de execuções de jobs
* `system.lakeflow.job_tasks` - Tasks individuais dos jobs
* `system.lakeflow.job_task_run_timeline` - Timeline de tasks
* `system.lakeflow.pipelines` - Pipelines (Spark Declarative)

### 📋 Significado:
Monitore orquestração de workflows: jobs, tasks, pipelines e seus históricos de execução, falhas e durações.

### ✅ Casos de Uso:
1. **Monitorar SLA de jobs críticos**
2. **Identificar jobs com falhas recorrentes**
3. **Análise de duração de tasks**
4. **Otimização de pipelines ETL**
5. **Alertas de falha em produção**

In [0]:
%sql
SELECT 
  j.job_id,
  j.name as job_name,
  j.creator_user_name as created_by,
  jrt.run_id,
  jrt.result_state,
  jrt.termination_code as state_message,
  jrt.error_message,
  jrt.period_start_time as start_time,
  jrt.period_end_time as end_time,
  ROUND((unix_timestamp(jrt.period_end_time) - unix_timestamp(jrt.period_start_time)) / 60, 2) as duracao_minutos
FROM 
  system.lakeflow.jobs j
  INNER JOIN system.lakeflow.job_run_timeline jrt 
    ON j.job_id = jrt.job_id 
    AND j.workspace_id = jrt.workspace_id
WHERE 
  jrt.period_start_time >= CURRENT_DATE() - INTERVAL 7 DAYS
  AND jrt.result_state IN ('FAILED', 'TIMED_OUT', 'CANCELED')
ORDER BY jrt.period_start_time DESC
LIMIT 50

## 6️⃣ SYSTEM.MLFLOW - Machine Learning 🤖

### Tabelas Disponíveis:
* `system.mlflow.experiments_latest` - Experimentos de ML
* `system.mlflow.runs_latest` - Runs (execuções) de treinamento
* `system.mlflow.run_metrics_history` - Métricas ao longo do tempo

### 📋 Significado:
Rastreie experimentos de Machine Learning: modelos treinados, métricas (accuracy, loss, etc.), parâmetros e artifacts.

### ✅ Casos de Uso:
1. **Comparar performance de modelos**
2. **Rastrear evolução de métricas**
3. **Governança de modelos em produção**
4. **Auditoria de treinamentos**
5. **Identificar melhores hiperparâmetros**

In [0]:
%sql
-- Melhores modelos por accuracy
SELECT 
  r.experiment_id,
  e.name as experiment_name,
  r.run_id,
  r.run_name,
  r.created_by as user_id,
  m.metric_name as metrica,
  m.metric_value as valor_metrica,
  r.start_time,
  r.status
FROM 
  system.mlflow.runs_latest r
  INNER JOIN system.mlflow.experiments_latest e ON r.experiment_id = e.experiment_id
  INNER JOIN system.mlflow.run_metrics_history m ON r.run_id = m.run_id
WHERE 
  m.metric_name = 'accuracy'
  AND r.status = 'FINISHED'
ORDER BY m.metric_value DESC
LIMIT 10;

## 7️⃣ SYSTEM.SERVING - Model Serving 🌐

### Tabelas Disponíveis:
* `system.serving.endpoint_usage` - Uso de endpoints de modelos
* `system.serving.served_entities` - Modelos servidos em produção

### 📋 Significado:
Monitore endpoints de modelos em produção: requisições, latência, throughput e versões deployadas.

### ✅ Casos de Uso:
1. **Monitorar latência de inferência**
2. **Análise de volume de requisições**
3. **Identificar modelos subutilizados**
4. **Alertas de degradação de performance**
5. **Planejamento de scaling**

In [0]:
-- Uso de endpoints nos últimos 7 dias
SELECT 
  se.endpoint_id,
  se.endpoint_name,
  DATE(eu.request_time) as data,
  COUNT(*) as total_requisicoes,
  ROUND(AVG(eu.usage_context['latency_ms']), 2) as latencia_media_ms,
  ROUND(MAX(eu.usage_context['latency_ms']), 2) as latencia_maxima_ms,
  SUM(CASE WHEN eu.status_code >= 400 THEN 1 ELSE 0 END) as erros
FROM 
  system.serving.endpoint_usage eu
  LEFT JOIN system.serving.served_entities se
    ON eu.served_entity_id = se.served_entity_id
WHERE 
  eu.request_time >= CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY 1, 2, 3
ORDER BY 3 DESC, 4 DESC;

## 8️⃣ SYSTEM.AI_GATEWAY - GenAI & LLMs 🧠

### Tabelas Disponíveis:
* `system.ai_gateway.usage` - Uso de AI Gateway (LLMs externos)

### 📋 Significado:
Rastreie chamadas a modelos de linguagem (OpenAI, Anthropic, etc.) através do AI Gateway: tokens consumidos, custos, latência.

### ✅ Casos de Uso:
1. **Controlar custos de APIs de LLM**
2. **Monitorar consumo de tokens**
3. **Análise de latência de inferência**
4. **Auditoria de uso por usuário/app**
5. **Otimizar prompts (reduzir tokens)**

In [0]:
%sql
-- Análise de uso de LLMs
SELECT 
  DATE(event_time) as data,
  endpoint_name,
  destination_model,
  COUNT(*) as total_requests,
  SUM(input_tokens) as tokens_input,
  SUM(output_tokens) as tokens_output,
  SUM(total_tokens) as tokens_totais,
  ROUND(AVG(latency_ms), 2) as latencia_media_ms
FROM 
  system.ai_gateway.usage
WHERE 
  event_time >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY 1, 2, 3
ORDER BY 1 DESC, tokens_totais DESC;

## 9️⃣ SYSTEM.STORAGE - Otimização de Armazenamento 💾

### Tabelas Disponíveis:
* `system.storage.predictive_optimization_operations_history` - Histórico de otimizações automáticas

### 📋 Significado:
Rastreie operações de Predictive Optimization (auto-compaction, auto-clustering, etc.) que o Databricks executa automaticamente nas suas tabelas.

### ✅ Casos de Uso:
1. **Verificar otimizações automáticas**
2. **Monitorar redução de arquivos pequenos**
3. **Avaliar impacto de clustering**
4. **Troubleshoot problemas de performance**
5. **Entender overhead de manutenção**

In [0]:
%sql
-- Histórico de Predictive Optimization
SELECT 
  catalog_name as table_catalog,
  schema_name as table_schema,
  table_name,
  operation_type,
  operation_status,
  start_time,
  end_time,
  operation_metrics['num_files_added'] as files_added,
  operation_metrics['num_files_removed'] as files_removed,
  ROUND((unix_timestamp(end_time) - unix_timestamp(start_time)) / 60, 2) as duracao_minutos
FROM 
  system.storage.predictive_optimization_operations_history
WHERE 
  start_time >= CURRENT_DATE() - INTERVAL 7 DAYS
ORDER BY start_time DESC
LIMIT 50;

## 🔟 SYSTEM.AI - AI Assistant (Schema vazio no momento)

### Tabelas Disponíveis:
Atualmente não há tabelas públicas neste schema.

### 📋 Significado:
Reservado para funcionalidades futuras relacionadas ao AI Assistant do Databricks.

---

## 1️⃣1️⃣ SYSTEM.INFORMATION_SCHEMA - Metadados do Unity Catalog 📋

### Significado:
Segue o padrão ANSI SQL para metadados. Contém informações sobre:
* Catálogos, schemas e tabelas
* Colunas e tipos de dados
* Views e funções
* Permissões e grants

### ✅ Casos de Uso:
1. **Descoberta de dados (data discovery)**
2. **Documentação automática de schemas**
3. **Auditoria de permissões**
4. **Governança de metadados**
5. **Inventário de objetos do Unity Catalog**

In [0]:
%sql
-- Listar todas as tabelas de um catálogo específico
SELECT 
  table_catalog,
  table_schema,
  table_name,
  table_type,
  comment
FROM 
  system.information_schema.tables
WHERE 
  table_catalog = 'curso_databricks'
ORDER BY table_schema, table_name;

## 📊 Resumo: Quando Usar Cada Schema

| Schema | Use quando precisar... |
|--------|------------------------|
| **billing** | 💰 Analisar custos, DBUs consumidos, chargeback |
| **access** | 🔐 Auditoria, compliance, linhagem de dados, segurança |
| **query** | 🚀 Otimizar performance, identificar queries lentas |
| **compute** | 🖥️ Monitorar clusters, warehouses, recursos computacionais |
| **lakeflow** | 🔄 Acompanhar jobs, pipelines, orquestração de workflows |
| **mlflow** | 🤖 Rastrear experimentos de ML, comparar modelos |
| **serving** | 🌐 Monitorar endpoints de modelos em produção |
| **ai_gateway** | 🧠 Controlar custos de LLMs, tokens consumidos |
| **storage** | 💾 Verificar otimizações automáticas de tabelas |
| **information_schema** | 📋 Descoberta de metadados, inventário do Unity Catalog |

---

### 🎯 Dica Final:
Combine múltiplos schemas em uma única análise!

**Exemplo:** Cruzar `system.query.history` (performance) com `system.billing.usage` (custos) para identificar queries caras e lentas.

In [0]:
%sql
-- Combinar dados de query history + billing para análise completa
WITH queries_lentas AS (
  SELECT 
    statement_id,
    executed_by,
    compute.warehouse_id as warehouse_id,
    total_duration_ms,
    read_bytes,
    DATE(start_time) as data_execucao
  FROM system.query.history
  WHERE 
    start_time >= CURRENT_DATE() - INTERVAL 7 DAYS
    AND total_duration_ms > 30000 -- > 30 segundos
),
custos AS (
  SELECT 
    usage_date,
    usage_metadata.warehouse_id as warehouse_id,
    SUM(usage_quantity) as dbus_consumidos
  FROM system.billing.usage
  WHERE usage_date >= CURRENT_DATE() - INTERVAL 7 DAYS
    AND usage_metadata.warehouse_id IS NOT NULL
    AND usage_unit = 'DBU'
  GROUP BY 1, 2
)
SELECT 
  q.statement_id,
  q.executed_by,
  q.warehouse_id,
  ROUND(q.total_duration_ms / 1000, 2) as tempo_segundos,
  ROUND(q.read_bytes / 1024 / 1024, 2) as mb_lidos,
  c.dbus_consumidos,
  q.data_execucao
FROM queries_lentas q
LEFT JOIN custos c 
  ON q.data_execucao = c.usage_date 
  AND q.warehouse_id = c.warehouse_id
ORDER BY q.total_duration_ms DESC
LIMIT 20;